# 16 · High-resolution videos for the README

Renders, at 512 px and from three cameras (the policy's own view `agentview`, `frontview`, `sideview`):
* **robot demos** generated from one of my recordings (object-relative retargeting with the planned approach), and
* **the learned policy** (`act_ours_all_vertical_s0`: all 10 demos, approach planned; 98% success) in evaluation scenes it never saw.

The policy receives exactly its usual input (cameras rendered at 256 px, resized to 128 px); the high-resolution frames come from a separate render of the same cameras (`tools/render_hero.py`).

**Runtime:** T4, about 15 min. Results go to the Hub (`results_zips/hero_videos.zip`) and are also offered as a download, so the notebook can finish unattended (set `AUTO_DISCONNECT = True` to release the GPU at the end).

In [ ]:
import os, sys, json, subprocess
os.environ["MUJOCO_GL"] = "egl"                      # GPU rendering, must be set before mujoco is imported
REPO_URL = "https://github.com/LukasWill/phone2panda.git"
BASE = "/content" if os.path.exists("/content") else "/kaggle/working"   # Colab or Kaggle
CODE_DIR = f"{BASE}/phone2panda"
if not os.path.exists(CODE_DIR):
    subprocess.run(["git", "clone", "-q", REPO_URL, CODE_DIR], check=True)
else:
    subprocess.run(["git", "-C", CODE_DIR, "pull", "-q"], check=True)
# LIBERO needs egl_probe / hf-egl-probe, which are built from source with cmake. pip's isolated build cannot run
# the pip-installed cmake wrapper, and the old egl_probe CMakeLists is rejected by cmake 4: build both first,
# without isolation and with the CMake 3.5 policy floor.
os.environ["CMAKE_POLICY_VERSION_MINIMUM"] = "3.5"
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "setuptools", "wheel", "cmake>=3.29,<4.2"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-build-isolation", "egl_probe", "hf-egl-probe"], check=True)
r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "lerobot[libero]==0.6.1", "imageio-ffmpeg"],
                   capture_output=True, text=True)
if r.returncode != 0:                                  # show pip's own reason, not just "exit status 1"
    print((r.stdout + r.stderr)[-4000:]); raise RuntimeError("pip install failed: see the lines above")
# Colab preinstalls an older torchao; a newer diffusers (pulled in by lerobot) then fails on import
# (FqnToConfig). ACT does not use torchao, and diffusers skips it when it is absent.
subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "torchao"], check=False)
os.chdir(CODE_DIR); sys.path.insert(0, CODE_DIR)
from p2p.sim.env import ensure_nvidia_egl
ensure_nvidia_egl()      # Colab: make EGL render on the GPU (otherwise it silently renders on the CPU, ~10x slower)
# Hugging Face token (Colab: key icon -> Secrets -> HF_TOKEN; Kaggle: Add-ons -> Secrets -> HF_TOKEN)
try:
    from google.colab import userdata; os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
except Exception:
    try:
        from kaggle_secrets import UserSecretsClient; os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    except Exception:
        print("No HF_TOKEN found: uploads to the Hub will fail")
if os.environ.get("HF_TOKEN"):
    from huggingface_hub import whoami
    print("Hugging Face account:", whoami(token=os.environ["HF_TOKEN"])["name"])
import torch; print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")

In [ ]:
HUB_REPO = "phone2panda-data"
RUN = "act_ours_all_vertical_s0"
INITS = [0, 1, 2, 3, 4, 5, 6, 7]
DEMO = "92e0b86bc6_lefthand"
SIZE = 512
AUTO_DISCONNECT = False
OUT = f"{BASE}/hero_videos"
from p2p.hub import pull
pull(HUB_REPO, f"{BASE}/hub", allow_patterns=[f"runs/{RUN}/final/*"])

In [ ]:
INITS_ARG = " ".join(map(str, INITS))
!python tools/render_hero.py {OUT} --ckpt {BASE}/hub/runs/{RUN}/final --inits {INITS_ARG} --demo {DEMO} --size {SIZE}
print(open(f"{OUT}/summary.json").read())

In [ ]:
# Finish: upload to the Hub first (works unattended), then offer a download; optionally release the GPU.
import os, traceback
from huggingface_hub import HfApi
from p2p.hub import full_repo_id
try:
    !cd {BASE} && zip -qr hero_videos.zip hero_videos
    repo = full_repo_id(HUB_REPO)
    HfApi(token=os.environ["HF_TOKEN"]).upload_file(path_or_fileobj=f"{BASE}/hero_videos.zip",
        path_in_repo="results_zips/hero_videos.zip", repo_id=repo, repo_type="dataset")
    print("uploaded:", f"https://huggingface.co/datasets/{repo}/blob/main/results_zips/hero_videos.zip")
    try:
        from google.colab import files; files.download(f"{BASE}/hero_videos.zip")
    except Exception:
        pass
except Exception:
    traceback.print_exc()
finally:
    if AUTO_DISCONNECT:
        from google.colab import runtime
        runtime.unassign()